# GPT-2 Instruction Fine-Tuning (Supervised Fine-Tuning / SFT)

This notebook demonstrates how to instruction fine-tune a **GPT-2 Medium (355M)** model into an **instruction-following assistant** capable of answering questions, transforming sentences, correcting grammar, and executing user commands.

### Pipeline Overview
1. **Architecture Setup**: GPT-2 Medium 355M configuration (24 layers, 16 heads, 1024 embedding dimension).
2. **Load Pretrained Weights**: Reusing existing local GPT-2 355M weights without redownloading.
3. **Alpaca Prompt Formatting**: Formatting instructions and context into prompt-response pairs.
4. **Custom Collate Function**: Padding variable-length sequences with `<|endoftext|>` and masking prompt loss with `-100`.
5. **Causal Language Model Fine-Tuning**: Autoregressive training on instruction-response pairs.
6. **Loading Fine-Tuned Checkpoint**: Reusing `gpt2-medium355M-sft.pth` if already trained.
7. **Interactive Assistant Testing**: Testing grammar correction, text rewriting, math conversions, and factual Q&A.

## 1. Environment & GPT-2 Medium Configurations

In [ ]:
import os
import math
import json
import time
from functools import partial

import numpy as np
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch version: {torch.__version__}")

# GPT-2 Medium (355M) configuration
GPT_CONFIG_355M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1024,
    "n_heads": 16,
    "n_layers": 24,
    "drop_rate": 0.0,
    "qkv_bias": True
}

## 2. Core GPT-2 Architecture (From Scratch)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=True):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.head_dim), dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"], context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

## 3. Loading GPT-2 Medium 355M Weights (Reusing Local Files)

In [ ]:
from gpt_download import download_and_load_gpt2

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(gpt.trf_blocks[b].att.out_proj.weight, params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(gpt.trf_blocks[b].att.out_proj.bias, params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(gpt.trf_blocks[b].ff.layers[0].weight, params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(gpt.trf_blocks[b].ff.layers[0].bias, params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(gpt.trf_blocks[b].ff.layers[2].weight, params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(gpt.trf_blocks[b].ff.layers[2].bias, params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

# Check for existing fine-tuned SFT weights
sft_checkpoint = os.path.join("..", "4. Fine-Tuning", "gpt2-medium355M-sft.pth")
model = GPTModel(GPT_CONFIG_355M)

if os.path.exists(sft_checkpoint):
    model.load_state_dict(torch.load(sft_checkpoint, map_location=device))
    print(f"Loaded fine-tuned SFT weights from: {sft_checkpoint}")
else:
    settings, params = download_and_load_gpt2(model_size="355M", models_dir="gpt2")
    load_weights_into_gpt(model, params)
    print("Loaded base GPT-2 355M weights.")

model.to(device)
tokenizer = tiktoken.get_encoding("gpt2")

## 4. Dataset & Alpaca-Style Prompt Formatting

We use the Alpaca prompt template to structure inputs:

In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input", "").strip() else ""
    return instruction_text + input_text

# Load instruction dataset
json_path = os.path.join("..", "4. Fine-Tuning", "instruction-data.json")
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total instruction examples: {len(data)}")
sample_prompt = format_input(data[0])
print("Sample Formatted Prompt:\n", sample_prompt)
print("Sample Output:\n", data[0]['output'])

### Custom Collate Function for Padding and Loss Masking

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

# Split dataset: 85% train, 5% val, 10% test
train_data = data[:int(len(data)*0.85)]
val_data = data[int(len(data)*0.85):int(len(data)*0.90)]
test_data = data[int(len(data)*0.90):]

customized_collate = partial(custom_collate_fn, device=device, allowed_max_length=1024)
train_loader = DataLoader(InstructionDataset(train_data, tokenizer), batch_size=4, shuffle=True, drop_last=True, collate_fn=customized_collate)
val_loader = DataLoader(InstructionDataset(val_data, tokenizer), batch_size=4, shuffle=False, collate_fn=customized_collate)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 5. Text Generation & Evaluation Functions

In [ ]:
def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=50256):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token_id = torch.multinomial(probs, num_samples=1)
        else:
            next_token_id = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and next_token_id.item() == eos_id:
            break

        idx = torch.cat((idx, next_token_id), dim=1)
    return idx

def query_assistant(instruction, input_context="", model=model, tokenizer=tokenizer, device=device):
    prompt = format_input({"instruction": instruction, "input": input_context})
    prompt_with_response_header = prompt + "\n\n### Response:\n"
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(prompt_with_response_header, tokenizer).to(device),
        max_new_tokens=120,
        context_size=1024,
        temperature=0.0,
        eos_id=50256
    )
    full_text = token_ids_to_text(token_ids, tokenizer)
    response = full_text[len(prompt_with_response_header):].strip()
    return response

## 6. Interactive Assistant Testing

We test our fine-tuned instruction model across different tasks: rewriting with similes, grammar editing, unit conversions, and direct Q&A.

In [ ]:
test_cases = [
    {"instruction": "Rewrite the sentence using a simile.", "input": "The car is very fast."},
    {"instruction": "Edit the sentence for grammar.", "input": "He go to the park every day."},
    {"instruction": "Convert this sentence to passive voice:", "input": "The chef cooked a delicious meal."},
    {"instruction": "Convert 45 kilometers to meters.", "input": ""},
    {"instruction": "What is the melting point of iron?", "input": ""},
    {"instruction": "Suggest a more formal synonym for 'happy'.", "input": ""}
]

print("--- Interactive Instruction-Following Tests ---\n")
for case in test_cases:
    resp = query_assistant(case['instruction'], case['input'], model=model, tokenizer=tokenizer, device=device)
    print(f"[Instruction] {case['instruction']}")
    if case['input']:
        print(f"[Input]       {case['input']}")
    print(f"[Assistant]   {resp}\n{'-'*50}")